# 🔐 SAHF — Self-Adaptive Homomorphic Framework
## ML-Based Noise Prediction for Confidential AI Computation

---

**Project:** Self-Adaptive Homomorphic Framework with ML-Driven Noise Prediction  
**TRL Level:** 4-5 (Lab validated with real components)  
**Encryption:** TenSEAL CKKS Scheme (poly_modulus_degree=8192)  
**ML Model:** LSTM Neural Network for noise time-series forecasting  

---

### Notebook Contents
1. **Setup & Imports**
2. **FHE Engine Demonstration**
3. **Dataset Generation** — Real FHE operations
4. **Data Preprocessing** — Normalization + Sliding Windows
5. **LSTM Model Training** — Noise prediction
6. **Evaluation** — Predictions vs Actual
7. **Decision Engine** — Adaptive bootstrap scheduling
8. **Baseline vs Adaptive Comparison**
9. **Feedback Loop Simulation**
10. **Save Outputs**

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# TensorFlow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow import keras

# Project modules
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.utils import (
    logger, print_banner, ensure_directories, save_metrics,
    noise_growth_model, compute_noise_proxy, create_sliding_windows
)
from src.fhe_simulation import CKKSContextManager, generate_fhe_noise_dataset, TENSEAL_AVAILABLE
from src.model import build_lstm_model, DataPreprocessor, train_model, NoisePredictor
from src.decision_engine import (
    AdaptiveDecisionEngine, BaselineSystem, AdaptiveSystem,
    run_comparison, ACTION_CONTINUE, ACTION_PARTIAL, ACTION_FULL
)
from src.telemetry import TelemetryEngine, TelemetrySnapshot

# Plot style
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.facecolor': '#0f0c29',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#333366',
    'axes.grid': True,
    'grid.alpha': 0.15,
})

# Ensure directories
ROOT = os.path.join(os.getcwd(), '..')
ensure_directories(ROOT)

print_banner()
print(f'TensorFlow version: {tf.__version__}')
print(f'TenSEAL available: {TENSEAL_AVAILABLE}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

## 2. FHE Engine Demonstration

Demonstrate the CKKS encryption engine: vector encryption, homomorphic addition and multiplication, and noise tracking.

In [ ]:
# Create CKKS context
ctx = CKKSContextManager(poly_modulus_degree=8192)

# Encrypt two vectors
plain_a = [1.0, 2.0, 3.0, 4.0]
plain_b = [0.5, 1.5, 2.5, 3.5]

enc_a = ctx.encrypt_vector(plain_a)
enc_b = ctx.encrypt_vector(plain_b)

print(f'Encrypted vectors created')
print(f'Initial telemetry: {ctx.get_telemetry()}')

In [ ]:
# Homomorphic Addition
enc_sum = ctx.homomorphic_add(enc_a, enc_b)
result_add = ctx.decrypt_vector(enc_sum)

print(f'\n=== Homomorphic Addition ===')
print(f'  a = {plain_a}')
print(f'  b = {plain_b}')
print(f'  a + b = {[round(x, 4) for x in result_add]}')
print(f'  Expected: {[a+b for a,b in zip(plain_a, plain_b)]}')
print(f'  Noise after addition: {ctx.noise_estimate:.6f}')
print(f'  Telemetry: {ctx.get_telemetry()}')

In [ ]:
# Homomorphic Multiplication
enc_prod = ctx.homomorphic_multiply(enc_a, enc_b)
result_mul = ctx.decrypt_vector(enc_prod)

print(f'\n=== Homomorphic Multiplication ===')
print(f'  a = {plain_a}')
print(f'  b = {plain_b}')
print(f'  a × b = {[round(x, 4) for x in result_mul]}')
print(f'  Expected: {[a*b for a,b in zip(plain_a, plain_b)]}')
print(f'  Noise after multiplication: {ctx.noise_estimate:.6f}')
print(f'  Depth increased to: {ctx.depth}')
print(f'  Telemetry: {ctx.get_telemetry()}')

In [ ]:
# Demonstrate noise growth over multiple operations
print('\n=== Noise Growth Over Operations ===')
ctx.reset()

noise_trace = [ctx.noise_estimate]
ops_trace = ['init']

# Perform 20 mixed operations
a = ctx.encrypt_vector([1.0, 2.0, 3.0, 4.0])
b = ctx.encrypt_vector([0.5, 1.5, 2.5, 3.5])

for i in range(20):
    if np.random.random() > 0.4:
        a = ctx.homomorphic_add(a, b)
        ops_trace.append('add')
    else:
        a = ctx.homomorphic_multiply(a, b)
        ops_trace.append('mul')
    noise_trace.append(ctx.noise_estimate)
    print(f'  Step {i+1:2d} | Op: {ops_trace[-1]:3s} | Noise: {ctx.noise_estimate:.6f} | Depth: {ctx.depth}')

# Plot noise growth
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#4ade80' if op == 'add' else '#f87171' for op in ops_trace[1:]]
ax.plot(noise_trace, color='#667eea', linewidth=2, zorder=1)
ax.scatter(range(1, len(noise_trace)), noise_trace[1:], c=colors, s=40, zorder=2)
ax.axhline(y=0.85, color='#f87171', linestyle='--', alpha=0.5, label='Bootstrap threshold')
ax.axhline(y=0.50, color='#fbbf24', linestyle='--', alpha=0.5, label='Refresh threshold')
ax.set_title('Noise Growth During FHE Operations', color='#a8edea')
ax.set_xlabel('Operation #')
ax.set_ylabel('Noise Level')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_noise_growth_demo.png'))
plt.show()

## 3. Dataset Generation

Generate a realistic FHE noise dataset using actual homomorphic operations.

**Features:**
- `step` — operation index
- `op_type` — 0 (add) or 1 (multiply)
- `depth` — current circuit depth
- `scale` — normalized scale factor
- `delta_noise` — noise change from this operation
- `noise_ratio` — noise amplification ratio
- `since_last_reset` — steps since last bootstrap
- `noise` — current noise level

**Target:** `next_noise` — noise at the next step (regression)

In [ ]:
# Generate dataset
DATASET_PATH = os.path.join(ROOT, 'data', 'fhe_noise_dataset.csv')

df = generate_fhe_noise_dataset(
    num_steps=3000,
    save_path=DATASET_PATH,
    seed=42
)

print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Dataset statistics
print('=== Dataset Statistics ===')
df.describe()

In [ ]:
# Dataset visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('FHE Noise Dataset Overview', fontsize=16, color='#a8edea')

# Noise over time
axes[0, 0].plot(df['noise'], color='#667eea', linewidth=0.8, alpha=0.8)
axes[0, 0].set_title('Noise Over Time')
axes[0, 0].set_xlabel('Step')
axes[0, 0].set_ylabel('Noise Level')

# Next noise distribution
axes[0, 1].hist(df['next_noise'], bins=50, color='#764ba2', alpha=0.7, edgecolor='none')
axes[0, 1].set_title('Next Noise Distribution')

# Depth distribution
axes[0, 2].hist(df['depth'], bins=30, color='#4ade80', alpha=0.7, edgecolor='none')
axes[0, 2].set_title('Depth Distribution')

# Delta noise
axes[1, 0].plot(df['delta_noise'], color='#fbbf24', linewidth=0.5, alpha=0.7)
axes[1, 0].set_title('Delta Noise Per Step')

# Op type bar chart
op_counts = df['op_type'].value_counts()
axes[1, 1].bar(['Addition (0)', 'Multiply (1)'],
               [op_counts.get(0, 0), op_counts.get(1, 0)],
               color=['#4ade80', '#f87171'], alpha=0.8)
axes[1, 1].set_title('Operation Distribution')

# Correlation heatmap
corr = df[['op_type', 'depth', 'scale', 'delta_noise', 'noise_ratio', 'noise', 'next_noise']].corr()
sns.heatmap(corr, ax=axes[1, 2], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1, 2].set_title('Feature Correlation')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_dataset_overview.png'))
plt.show()

## 4. Data Preprocessing

1. **Normalize** features using MinMaxScaler
2. **Create sliding windows** (window_size = 60 steps)
3. **Train/Val/Test split** (70% / 15% / 15%)

In [ ]:
# Preprocessing
WINDOW_SIZE = 60

preprocessor = DataPreprocessor(window_size=WINDOW_SIZE)

feature_cols = ['op_type', 'depth', 'scale', 'delta_noise', 
                'noise_ratio', 'since_last_reset', 'noise']

data = preprocessor.prepare_data(df, feature_cols=feature_cols)

print('\nPreprocessed data shapes:')
for key, val in data.items():
    print(f'  {key}: {val.shape}')

print(f'\nInput shape for model: {data["X_train"].shape[1:]}')
print(f'  → Window size: {data["X_train"].shape[1]}')
print(f'  → Features: {data["X_train"].shape[2]}')

## 5. LSTM Model Training

### Architecture
```
Input(window_size, 7)
  → LSTM(64, return_sequences=True)
  → Dropout(0.2)
  → LSTM(32)
  → Dense(16, relu)
  → Dense(1)  [output: predicted next_noise]
```

### Training Config
- **Loss:** Huber (δ=1.0)
- **Optimizer:** Adam (lr=0.001)
- **Callbacks:** EarlyStopping (patience=15), ReduceLROnPlateau (patience=7)
- **Target:** MAE < 0.05

In [ ]:
# Build model
input_shape = (data['X_train'].shape[1], data['X_train'].shape[2])

model = build_lstm_model(input_shape)
model.summary()

In [ ]:
# Train model
MODEL_PATH = os.path.join(ROOT, 'models', 'lstm_model.keras')

training_results = train_model(
    model, data,
    epochs=100,
    batch_size=32,
    model_save_path=MODEL_PATH
)

print('\n=== Training Results ===')
for key in ['train_loss', 'train_mae', 'val_loss', 'val_mae', 'test_loss', 'test_mae', 'epochs_trained']:
    print(f'  {key}: {training_results[key]}')

In [ ]:
# Training history visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=16, color='#a8edea')

hist = training_results['history']

# Loss
axes[0].plot(hist['loss'], label='Train Loss', color='#667eea', linewidth=2)
axes[0].plot(hist['val_loss'], label='Val Loss', color='#fbbf24', linewidth=2)
axes[0].set_title('Loss (Huber)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# MAE
axes[1].plot(hist['mae'], label='Train MAE', color='#667eea', linewidth=2)
axes[1].plot(hist['val_mae'], label='Val MAE', color='#fbbf24', linewidth=2)
axes[1].axhline(y=0.05, color='#4ade80', linestyle='--', alpha=0.7, label='Target (0.05)')
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_training_history.png'))
plt.show()

if training_results['test_mae'] < 0.05:
    print(f'\n✅ TARGET MET! Test MAE = {training_results["test_mae"]:.6f} < 0.05')
else:
    print(f'\n⚠️  Test MAE = {training_results["test_mae"]:.6f} (target: < 0.05)')

## 6. Evaluation — Predictions vs Actual

In [ ]:
# Load best model
from tensorflow.keras.models import load_model

best_model = load_model(MODEL_PATH)
print(f'Best model loaded from {MODEL_PATH}')

# Predictions on test set
test_preds_scaled = best_model.predict(data['X_test'], verbose=0).flatten()
test_actual_scaled = data['y_test']

# Inverse transform
test_preds = preprocessor.inverse_transform_target(test_preds_scaled)
test_actual = preprocessor.inverse_transform_target(test_actual_scaled)

# Compute metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(test_actual, test_preds)
rmse = np.sqrt(mean_squared_error(test_actual, test_preds))
r2 = r2_score(test_actual, test_preds)

print(f'\n=== Test Set Metrics ===')
print(f'  MAE:  {mae:.6f}')
print(f'  RMSE: {rmse:.6f}')
print(f'  R²:   {r2:.6f}')

In [ ]:
# Predicted vs Actual plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation: Predicted vs Actual Noise', fontsize=16, color='#a8edea')

# Time series comparison
n_show = min(300, len(test_preds))
axes[0].plot(range(n_show), test_actual[:n_show], label='Actual',
             color='#4ade80', linewidth=1.5, alpha=0.8)
axes[0].plot(range(n_show), test_preds[:n_show], label='Predicted',
             color='#f87171', linewidth=1.5, alpha=0.8)
axes[0].set_title('Predicted vs Actual (Time Series)')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Noise Level')
axes[0].legend()

# Scatter
axes[1].scatter(test_actual, test_preds, alpha=0.3, s=10, color='#667eea')
lims = [min(test_actual.min(), test_preds.min()),
        max(test_actual.max(), test_preds.max())]
axes[1].plot(lims, lims, '--', color='#4ade80', alpha=0.7, label='Perfect')
axes[1].set_title(f'Scatter Plot (R² = {r2:.4f})')
axes[1].set_xlabel('Actual Noise')
axes[1].set_ylabel('Predicted Noise')
axes[1].legend()

# Error distribution
errors = np.abs(test_preds - test_actual)
axes[2].hist(errors, bins=50, color='#764ba2', alpha=0.7, edgecolor='none')
axes[2].axvline(x=np.mean(errors), color='#fbbf24', linestyle='--',
                linewidth=2, label=f'Mean: {np.mean(errors):.4f}')
axes[2].axvline(x=np.median(errors), color='#4ade80', linestyle='--',
                linewidth=2, label=f'Median: {np.median(errors):.4f}')
axes[2].set_title('Prediction Error Distribution')
axes[2].set_xlabel('Absolute Error')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_predictions_eval.png'))
plt.show()

## 7. Decision Engine — Adaptive Bootstrap Scheduling

### Decision Rules
```
if predicted_noise > 0.85:
    action = "FULL_BOOTSTRAP"     → Reset noise to 0.01
elif predicted_noise > 0.50:
    action = "PARTIAL_REFRESH"    → Reduce noise by 50%
else:
    action = "CONTINUE"           → No action needed
```

### Mathematical Model
$$N(t+1) = N(t) \times \big(1 + w_1 \cdot D + w_2 \cdot W_{op} + w_3 \cdot S_f\big)$$

$$O' = O \times \big(1 + w_1 \cdot \text{Depth} + w_2 \cdot \text{Op\_weight} + w_3 \cdot \text{Scale\_factor}\big)$$

In [ ]:
# Demonstrate decision engine
engine = AdaptiveDecisionEngine(
    high_threshold=0.85,
    medium_threshold=0.50
)

# Test with various noise levels
test_cases = [
    (0.10, 'Low noise'),
    (0.30, 'Moderate noise'),
    (0.55, 'Elevated noise'),
    (0.75, 'High noise'),
    (0.90, 'Critical noise'),
]

print('=== Decision Engine Demo ===')
print(f'{"Noise":>8} | {"Level":>16} | {"Action":>18}')
print('-' * 50)

for noise, desc in test_cases:
    action = engine.decide(predicted_noise=noise, step=0)
    emoji = '✅' if action == ACTION_CONTINUE else ('🟡' if action == ACTION_PARTIAL else '🔴')
    print(f'{noise:>8.2f} | {desc:>16} | {emoji} {action}')

print(f'\nDecision stats: {engine.get_stats()}')

## 8. Baseline vs Adaptive Comparison

**System A (Baseline):** Fixed-interval bootstrapping every N steps  
**System B (Adaptive):** ML-driven bootstrapping based on LSTM predictions  

Both process the **same sequence of operations** for fair comparison.

In [ ]:
# Run comparison
NUM_COMPARISON_STEPS = 500
BASELINE_INTERVAL = 20

# Create predictor with trained model
predictor = NoisePredictor(
    model_path=MODEL_PATH,
    preprocessor=preprocessor
)

comparison = run_comparison(
    num_steps=NUM_COMPARISON_STEPS,
    baseline_interval=BASELINE_INTERVAL,
    predictor=predictor,
    seed=42
)

# Display results
imp = comparison['improvement']
print('\n' + '='*60)
print('COMPARISON RESULTS')
print('='*60)
for k, v in imp.items():
    print(f'  {k}: {v}')
print('='*60)

In [ ]:
# Comparison visualization
b_noise = comparison['baseline']['noise_history']
a_noise = comparison['adaptive']['noise_history']
a_pred = comparison['adaptive'].get('prediction_history', [])
b_events = comparison['baseline']['bootstrap_events']
a_events = comparison['adaptive']['bootstrap_events']
actions_list = comparison['adaptive'].get('action_history', [])

steps = list(range(len(b_noise)))

fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                          gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('Baseline vs Adaptive: Noise Over Time', fontsize=16, color='#a8edea')

# Noise traces
axes[0].plot(steps, b_noise, label='Baseline Noise',
             color='#f87171', linewidth=1.2, alpha=0.7)
axes[0].plot(steps, a_noise, label='Adaptive Noise',
             color='#4ade80', linewidth=1.5)
if a_pred:
    axes[0].plot(steps[:len(a_pred)], a_pred, label='ML Prediction',
                 color='#fbbf24', linewidth=1, linestyle=':', alpha=0.6)

# Bootstrap markers
if b_events:
    axes[0].scatter(b_events,
                    [b_noise[min(e, len(b_noise)-1)] for e in b_events],
                    marker='x', s=40, color='#f87171', label='Baseline Bootstrap', zorder=5)
if a_events:
    axes[0].scatter(a_events,
                    [a_noise[min(e, len(a_noise)-1)] for e in a_events],
                    marker='D', s=40, color='#4ade80', label='Adaptive Bootstrap', zorder=5)

axes[0].axhline(y=0.85, color='#f87171', linestyle='--', alpha=0.3, label='Full Bootstrap (0.85)')
axes[0].axhline(y=0.50, color='#fbbf24', linestyle='--', alpha=0.3, label='Partial Refresh (0.50)')
axes[0].set_ylabel('Noise Level')
axes[0].legend(loc='upper left', fontsize=9)

# Actions
if actions_list:
    action_map = {ACTION_CONTINUE: 0, ACTION_PARTIAL: 1, ACTION_FULL: 2}
    color_map = {ACTION_CONTINUE: '#4ade80', ACTION_PARTIAL: '#fbbf24', ACTION_FULL: '#f87171'}
    action_vals = [action_map.get(a, 0) for a in actions_list]
    colors = [color_map.get(a, '#4ade80') for a in actions_list]

    axes[1].scatter(range(len(action_vals)), action_vals, c=colors, s=5, alpha=0.7)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['CONTINUE', 'PARTIAL', 'FULL'])
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Action')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_comparison.png'))
plt.show()

In [ ]:
# Performance bars
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Performance Comparison: Baseline vs Adaptive', fontsize=16, color='#a8edea')

categories = ['Baseline', 'Adaptive']
colors = ['#f87171', '#4ade80']

# Bootstraps
vals = [imp['baseline_bootstraps'], imp['adaptive_bootstraps']]
bars = axes[0].bar(categories, vals, color=colors, alpha=0.8, width=0.5)
axes[0].set_title(f'Total Bootstraps\n(↓{imp["bootstrap_reduction_pct"]:.1f}%)')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'{int(val)}', ha='center', va='bottom', fontweight='bold', color='#e0e0ff')

# Latency
vals = [imp['baseline_latency'], imp['adaptive_latency']]
bars = axes[1].bar(categories, vals, color=colors, alpha=0.8, width=0.5)
axes[1].set_title(f'Total Latency\n(↓{imp["latency_improvement_pct"]:.1f}%)')
axes[1].set_ylabel('Seconds')
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'{val:.1f}s', ha='center', va='bottom', fontweight='bold', color='#e0e0ff')

# Avg noise
vals = [imp['baseline_avg_noise'], imp['adaptive_avg_noise']]
bars = axes[2].bar(categories, vals, color=colors, alpha=0.8, width=0.5)
axes[2].set_title('Average Noise Level')
axes[2].set_ylabel('Noise')
for bar, val in zip(bars, vals):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold', color='#e0e0ff')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_performance_bars.png'))
plt.show()

## 9. Feedback Loop Simulation

The feedback loop:
1. Stores prediction vs actual noise pairs
2. Monitors prediction accuracy over time
3. Triggers retraining when error exceeds threshold

In [ ]:
# Simulate feedback loop
from src.telemetry import TelemetryEngine, TelemetrySnapshot
from src.utils import get_timestamp

telemetry = TelemetryEngine(feedback_window=100)

# Simulate 200 prediction-actual pairs
np.random.seed(42)

feedback_noise = []
feedback_preds = []
feedback_errors = []

base_noise = 0.01
for i in range(200):
    # Simulate noise growth
    op = 'multiply' if np.random.random() > 0.5 else 'add'
    base_noise = noise_growth_model(base_noise, op, depth=i%10, scale=2**30)
    
    # Simulated prediction (with some error)
    pred = base_noise + np.random.normal(0, 0.02)
    pred = np.clip(pred, 0, 1)
    
    actual = base_noise
    error = abs(pred - actual)
    
    feedback_noise.append(actual)
    feedback_preds.append(pred)
    feedback_errors.append(error)
    
    # Determine action
    if pred > 0.85:
        action = 'FULL_BOOTSTRAP'
        base_noise = 0.01  # Reset
    elif pred > 0.5:
        action = 'PARTIAL_REFRESH'
        base_noise *= 0.5
    else:
        action = 'CONTINUE'
    
    # Record in telemetry
    snapshot = TelemetrySnapshot(
        timestamp=get_timestamp(),
        step=i,
        noise_actual=actual,
        noise_predicted=pred,
        op_type=op,
        depth=i % 10,
        action_taken=action
    )
    telemetry.record(snapshot)
    telemetry.record_prediction(i, pred, actual, action)

# Feedback summary
feedback_summary = telemetry.get_feedback_summary()
print('=== Feedback Loop Summary ===')
for k, v in feedback_summary.items():
    print(f'  {k}: {v}')

print(f'\nShould retrain: {telemetry.should_retrain()}')
retrain_data = telemetry.get_retrain_data()
if retrain_data is not None:
    print(f'Retrain data available: {retrain_data.shape}')

In [ ]:
# Feedback loop visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Feedback Loop Analysis', fontsize=16, color='#a8edea')

# Prediction vs actual over time
axes[0].plot(feedback_noise, label='Actual', color='#4ade80', linewidth=1.5)
axes[0].plot(feedback_preds, label='Predicted', color='#f87171', linewidth=1, alpha=0.7)
axes[0].set_title('Predictions vs Actual Over Time')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Noise')
axes[0].legend()

# Rolling error
rolling_errors = pd.Series(feedback_errors).rolling(20).mean()
axes[1].plot(rolling_errors, color='#667eea', linewidth=2)
axes[1].axhline(y=0.08, color='#f87171', linestyle='--', alpha=0.7, label='Retrain threshold')
axes[1].set_title('Rolling Prediction Error (window=20)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Mean Abs Error')
axes[1].legend()

# Error histogram
axes[2].hist(feedback_errors, bins=30, color='#764ba2', alpha=0.7, edgecolor='none')
axes[2].axvline(x=np.mean(feedback_errors), color='#fbbf24', linestyle='--',
                label=f'Mean: {np.mean(feedback_errors):.4f}')
axes[2].set_title('Error Distribution')
axes[2].set_xlabel('Absolute Error')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'plots', 'nb_feedback_loop.png'))
plt.show()

## 10. Save All Outputs

In [ ]:
# Compile final metrics
comp_clean = {
    'improvement': comparison['improvement'],
    'baseline': {
        'total_steps': comparison['baseline']['total_steps'],
        'total_bootstraps': comparison['baseline']['total_bootstraps'],
        'total_latency': comparison['baseline']['total_latency'],
        'avg_noise': comparison['baseline']['avg_noise'],
    },
    'adaptive': {
        'total_steps': comparison['adaptive']['total_steps'],
        'total_bootstraps': comparison['adaptive']['total_bootstraps'],
        'total_latency': comparison['adaptive']['total_latency'],
        'avg_noise': comparison['adaptive']['avg_noise'],
    }
}

all_metrics = {
    'project': 'SAHF - Self-Adaptive Homomorphic Framework',
    'trl_level': '4-5',
    'model': training_results,
    'comparison': comp_clean,
    'feedback': feedback_summary,
    'config': {
        'dataset_steps': 3000,
        'window_size': WINDOW_SIZE,
        'epochs': 100,
        'batch_size': 32,
        'comparison_steps': NUM_COMPARISON_STEPS,
        'baseline_interval': BASELINE_INTERVAL
    }
}

METRICS_PATH = os.path.join(ROOT, 'results', 'metrics.json')
save_metrics(all_metrics, METRICS_PATH)

print('\n' + '='*60)
print('🎉 ALL OUTPUTS SAVED SUCCESSFULLY!')
print('='*60)
print(f'  📊 Dataset:  {DATASET_PATH}')
print(f'  🧠 Model:    {MODEL_PATH}')
print(f'  📋 Metrics:  {METRICS_PATH}')
print(f'  📈 Plots:    {os.path.join(ROOT, "results", "plots")}')
print(f'\n  Model Test MAE:       {training_results["test_mae"]:.6f}')
print(f'  Bootstrap Reduction:  {comparison["improvement"]["bootstrap_reduction_pct"]:.1f}%')
print(f'  Latency Improvement:  {comparison["improvement"]["latency_improvement_pct"]:.1f}%')
print('='*60)
print('\n🚀 Run dashboard: streamlit run dashboard/app.py')